# 概述
在 create_agent() 的底层运行机制中，有几个重要的组件，分别是：
- 模型(Model) ：Agent 的“大脑”，负责理解任务与决策推理。
- 工具(Tools) ：Agent 的“手脚”，执行模型自己做不到的外部操作。
- 系统提示词(System Prompt) ：Agent的“角色”，告诉模型该怎么想、参考什么上下文。
- 中间件(Middleware) ：Agent的“中枢”，在执行流程的关键节点进行拦截、控制和增强。

## 什么是中间件
简单说就是Agent 执行过程中的钩子函数，是 LangChain 1.x 的“王牌”工程化
能力。
开发者可以高度定制和控制Agent运行的每一个环节，是处理 Agent 生命周期的标准方式。

中间件的价值就在于把这些与业务无关、但与执行过程强相关的横切逻辑，从 Agent 主流程中分离出来。

让Agent 主体代码 聚焦业务 ，而借助中间件，实现“ 拦截流程、修改流程、增强流程 ”。

## 中间件的分类
- 自定义中间件：开发者自定义
- 内置中间件： 
  - 模型供应商定制的中间件
  - 和模型供应商无关的中间件
### 和模型供应商无关的内置中间件
#### 成本与资源控制类
目标：控成本、控配额、避免无限调用，解决“agent太贵 太能跑 停不下来”的问题。

适合生产环境的成本治理、配额治理、长会话优化、SaaS 产品控费。

包含：

- Model call limit：限制模型调用次数，防止一次任务反复请求 LLM，导致费用失控。
  
- Tool call limit：限制工具调用次数，避免 Agent 无限试错、死循环调工具。
  
- Summarization：在上下文快满时自动总结历史，减少 token 消耗

- Context editing：裁剪上下文、清理工具调用痕迹，本质上也是为了节省上下文成本

#### 稳定性与容错保障类
核心目标：保证服务不中断、失败后尽量自动恢复。主要解决“ 调用失败怎么办、模型挂了怎么办、工具超时怎么办 ”。

适合线上生产系统，尤其是多模型、多工具依赖的 Agent。

本质上是在做 高可用、容灾、鲁棒性建设。

包含：

- Model fallback: 主模型失败时切换备用模型
- Model retry: 模型调用失败后自动重试
- Tool retry: 工具调用失败后自动重试

#### 安全与合规风控类
核心目标：让 Agent 可控、可审、合规。主要解决“ Agent乱执行、泄露敏感信息、做危险操作 ”的问题。

适合企业内部系统、客服系统、审批流、数据查询类 Agent。

尤其是涉及：发邮件、调数据库、调财务/人事系统、导出敏感信息、执行外部动作等

包含：
- Human-in-the-loop：在关键工具调用前暂停，等人工审批
- PII detection：检测和处理个人敏感信息
- Model call limit / Tool call limit：某种意义上也可归到风控，因为它能防止异常滥用
